# TP21 — Orchestration & outils

**25 novembre · jalon J5 · ½ journée après le module 6.4**

Notebook de travail à compléter. Source pédagogique : [Cahier_TP_NutriScope.md, TP21, lignes 669–685](../../../docs/Cahier_TP_NutriScope.md).

À l’issue du TP, l’assistant choisit entre `chercher_produit`, `substituer` et `repondre_source` dans LangChain. Il enchaîne « ce produit est-il bon ? → pourquoi ? → propose mieux → et en bio ? », conserve une mémoire courte et trace les outils, la latence et les jetons à chaque échange. Livrables du cahier : assistant orchestré, scénarios rejouables, journalisation active et revue croisée de dix minutes avec remarques en issues.

## 1. Choisir la base d’intégration

**Recommandation : partir du `develop` courant du dépôt d’équipe pour la future branche TP21 et prendre la PR #71 comme base technique.** Conserver l’outil produit, le retrieval et les données d’évaluation de #71. Réutiliser le chargement Parquet de #70 derrière un adaptateur quand le catalogue complet est nécessaire, ainsi que le principe de mémoire à échanges complets et les cas de test pertinents. Décision à valider avant de créer la branche.

| Sujet | PR #70 · `d35e6b0` | PR #71 · `f5960b7` | Choix TP21 |
|---|---|---|---|
| Catalogue | DataFrame issu de `products.parquet` et `nutrients.parquet`. Recherche par mots dans nom et marque, tri exact puis complétude. Fiche sans champ `base`. | Lot CSV d’atelier, liste de dictionnaires. Recherche par nom, repli marque, code prioritaire. La fiche expose `base` et conserve les inconnues. | API d’outil #71 pour démarrer. Si adaptation depuis #70, normaliser les champs et vérifier la base par produit avant de la déclarer. |
| Mémoire et clients | `repondre(question, catalogue, client, …, historique=None)`. Fenêtre de trois échanges précédents complets. Adaptateurs Ollama/OpenRouter et mesures par appel. | `repondre(question, client=None, catalogue=None)`. Conversation neuve par appel, boucle compatible OpenAI et mesures agrégées. | Une orchestration LangChain. Reprendre l’idée de fenêtre de #70 et conserver l’identifiant produit hors des messages tronqués. |
| RAG et évaluation | Assistant sans retrieval. Garde-fous, mesures de référence sans RAG et documentation. | `rechercher()` sur Chroma, corpus et 30 questions d’évaluation. Index opérationnel sur les fiches, génération citée absente dans ce snapshot. | Réutiliser `src/rag`, le corpus et les fixtures d’évaluation #71. Tester les citations et le passage d’un grade renseigné à inconnu. |
| Intégration | Ouverte, GitHub la signale non fusionnable lors de la vérification. Contient aussi le scanner. | Ouverte, GitHub la signale fusionnable lors de la vérification. L’audit a relevé un test Chroma en échec. | Vérifier les deux diffs contre `develop` au moment d’intégrer. Arbitrer explicitement les fichiers `src/assistant/*` en commun. |

Les deux branches exposent les mêmes arguments publics de recherche, mais les signatures Python et les représentations de catalogue diffèrent. Conserver les tests de garde-fous utiles de #70 et revoir les règles du prompt qui supposent un seul outil ou permettent au modèle de choisir une fiche ambiguë. Le vérificateur de chiffres peut signaler une anomalie. Il ne prouve ni l’identité du produit ni la validité d’une citation.

[PR #70](https://github.com/SDMM27/NutriScope/pull/70) · [PR #71](https://github.com/SDMM27/NutriScope/pull/71) · [Contrat du retrieval #71](https://github.com/clement-welsch/NutriScope/blob/f5960b7/src/rag/recherche.py)

## 2. Contrats proposés pour les outils

Les schémas refusent les champs inconnus. Les dépendances (catalogue, collection, client, paramètres de distance) sont injectées par l’application. Le modèle fournit les arguments métier. Les erreurs de validation et de service sont distinguées des recherches sans résultat. Les codes-barres restent des chaînes.

### `chercher_produit(nom=None, code=None, limite=3)` · TP18

- Au moins un nom ou un code non vide, et un entier strict de 1 à 5. Si le code est fourni, recherche exacte par code.
- Utiliser `chercher_produit(arguments, catalogue=…)` de #71. Variante #70 sous adaptateur : `chercher_produit(catalogue, arguments)`.
- Résultat existant : `{statut, total, produits}` avec `ok`, `ambigu` ou `introuvable`. `total` est avant troncature. Une fiche contient `code, nom, marque, base, nutriscore, nutriments, donnees_incompletes`. Arguments invalides : `{erreur}`. Intercepter séparément les erreurs de lecture à la frontière LangChain.
- En cas d’ambiguïté, présenter les candidats et demander une confirmation. Enregistrer le code dans la session après résolution certaine. Garder chaque inconnue à `null`, même pour un produit confirmé.

### `substituer(code, bio_uniquement=False, limite=3)` · TP16

- `code` correspond au produit de référence confirmé. Limite entière 1–5, indicateur bio booléen. « Et en bio ? » conserve le produit de référence. Le paramètre `bio_uniquement=True` exprime une demande explicite.
- Adaptateur proposé autour du futur `suggest_substitutes` du [support TP16](../../../docs/tps/tp16_segmentation_substitution.ipynb). Préparer une projection avec rayon, sous-catégorie, grade OFF, nutriments et base comparables, et statut bio à trois valeurs. Le lot #71 n’expose pas la sous-catégorie ni le bio. Le seuil de distance et les variables sont fixés côté service.
- Résultat proposé : `{statut, code_reference, substituts}`. Chaque candidat expose `{produit, bio, distance}`, avec `produit` au format fiche. Statuts `ok`, `introuvable`, `donnees_insuffisantes`, `aucun_substitut`. Erreur d’entrée : `{erreur}`.
- Exclure le produit source, les candidats d’un autre rayon ou d’une autre sous-catégorie, les distances non calculables sur les données réelles et les grades inconnus ou égaux. Trier par meilleur grade, puis distance, puis code pour départager. Respecter l’éventuelle priorité bio du produit source dans les règles du TP16. Sous filtre strict, seuls les candidats bio confirmés passent. Aucun élargissement silencieux si la liste est vide.

### `repondre_source(question, code=None, k=5)` · TP19–20

- Question non vide, contexte produit optionnel confirmé, entier strict `k` de 1 à 20. Le filtre d’index est construit côté application, à partir des métadonnées réellement indexées.
- Réutiliser `rechercher(question, k=…, filtres=…)` de #71 et préserver `id, score, texte, metadonnees`. Pour « pourquoi ? », joindre la fiche du code confirmé et les passages généraux pertinents. Filtrer toutes les sources par code ferait disparaître ces passages.
- Résultat proposé : `{statut, reponse, sources}`. Chaque source `{id, titre, url, extrait}` est issue d’un passage réellement récupéré ou d’une fiche utilisée. Statuts `ok`, `sources_insuffisantes`. Arguments invalides : `{erreur}`. Un service indisponible remonte une erreur distincte.
- Implémentation à faire pendant le TP : construire la réponse à partir des preuves, puis vérifier que les identifiants cités appartiennent aux sources retournées. En cas de sources insuffisantes, l’expliquer. Une fiche et un document général ne suffisent pas toujours à reconstituer le calcul officiel du produit. Corriger le grade périmé avant validation sur index persistant.

## 3. Préparer le premier essai

1. Relever la version LangChain utilisée dans le module 6.4 et la version Python du noyau.
2. Rendre les services injectables. Commencer par la recherche produit. Pour les deux outils manquants, utiliser des fixtures explicitement marquées pour tester le routage.
3. Ouvrir une session avec un identifiant stable, un code produit confirmé et des échanges récents complets. Laisser les résultats d’outils liés à leurs appels lorsque l’historique est tronqué.
4. Borner les appels et enregistrer une trace par échange, même si aucun outil ne part. Mesurer la durée totale, les latences outils et l’usage du modèle. En cas d’usage partiellement remonté, garder l’indisponibilité explicite au lieu de déclarer zéro.

**Point de contrôle.** Avant de brancher un modèle, expliquer où vit le code du produit confirmé et à quel moment il change.

In [ ]:
from importlib.util import find_spec
from pathlib import Path
import sys

racine = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "docs" / "Cahier_TP_NutriScope.md").is_file()),
    None,
)
print("Python :", sys.version.split()[0])
print("Dépôt :", racine)
print("LangChain disponible dans le noyau :", find_spec("langchain") is not None)

## 4. Scénario à rejouer

Dans le premier tour, renseigner un code présent dans le lot pilote. Avant le test complet, vérifier que ce lot permet au moins un meilleur candidat comparable et une option bio, et que le corpus contient une référence exploitable.

**Exercice 1.** Adapter la recherche et vérifier le tour 1. Quelle clé doit figurer dans la session à la sortie ?  
**Exercice 2.** Tester les tours 2 à 4 avec des fixtures. Vérifier que le référent reste stable et que le filtre bio apparaît uniquement au dernier tour.  
**Exercice 3.** Brancher les deux services réels, rejouer la série et examiner la trace et les citations. Un parcours simulé valide le routage seulement.

In [ ]:
def scenario_tp21(code_produit: str) -> list[dict]:
    """Décrit les entrées et les vérifications du scénario d'acceptation."""
    return [
        {
            "question": f"Ce produit est-il bon ? Son code-barres est {code_produit}.",
            "outil_attendu": "chercher_produit",
            "controle": "Fiche du code confirmé et valeurs concordantes avec le catalogue.",
        },
        {
            "question": "Pourquoi ?",
            "outil_attendu": "repondre_source",
            "controle": "Même produit, explication étayée et identifiants de sources vérifiables.",
        },
        {
            "question": "Propose mieux.",
            "outil_attendu": "substituer",
            "controle": "Même référence, produit comparable et mieux noté.",
        },
        {
            "question": "Et en bio ?",
            "outil_attendu": "substituer",
            "controle": "Même référence, bio_uniquement=True et bio confirmé.",
        },
    ]


# Exemple issu du lot TP18. Vérifier qu'il convient aux quatre tours avant la démo.
for numero, tour in enumerate(scenario_tp21("3017620422003"), start=1):
    print(numero, tour["question"], "→", tour["outil_attendu"])

### Cas de contrôle

| Cas | Attendu |
|---|---|
| Recherche ambiguë, puis « pourquoi ? » | Demander de préciser le produit. Aucun code arbitraire dans la mémoire. |
| Grade ou valeur nutritionnelle inconnue | Garder `null` dans la fiche et expliquer que l’information manque. |
| Aucun meilleur candidat bio | Liste vide et message explicite. |
| Index vide, source absente ou identifiant inventé | Réponse d’insuffisance ou rejet de la citation. |
| Nouvelle session après le premier parcours | « Et en bio ? » demande un produit de référence. |
| Historique tronqué | Appels et résultats restent appariés. Le produit confirmé reste accessible. |
| Modèle sans usage ou tour sans appel modèle | Jetons manquants signalés ; zéro uniquement si aucun jeton n’a été consommé. |

### Trace à compléter pendant le TP

Documenter par échange `session_id`, `tour`, `latence_ms`, `outils[]` (nom, statut, latence), `tokens_entree`, `tokens_sortie` et `statut`. Agréger les usages des appels réellement effectués. Identifier les scénarios simulés dans le rapport de rejeu.

In [ ]:
# À compléter pendant le TP : remplacer la liste vide par les traces du rejeu.
traces = []
print("Échanges enregistrés :", len(traces))

## 5. Revue croisée et rendu

Préparer dix minutes de conversation libre avec une autre équipe. Lui faire demander une explication, un substitut, une variante bio, puis essayer une ambiguïté ou une information absente. Pour chaque anomalie : question exacte, état de départ, attendu, résultat observé, référence de la trace. Consigner les remarques en issues pendant la séance.

- [ ] Assistant LangChain, mémoire courte et trois outils réels.
- [ ] Scénarios rejouables avec résultats et provenance des sources.
- [ ] Trace active à chaque échange.
- [ ] Revue croisée effectuée et remarques consignées.